## Gemini Master Instructions for Modular Colab **Workflows**

You are helping design and maintain **modular Google Colab workflows** orchestrated by a **central master pipeline**. Every notebook step must remain self-contained, reusable, and easy to debug independently, while still fitting into a larger end-to-end workflow. Make sure to read and follow the guidance below.

---

### 1. Core Architecture Principles

* Build workflows as **modular processing blocks**.
* Each major step must be **self-contained** with clear inputs/outputs.
* Use a **single master pipeline block** to orchestrate execution.
* Avoid tight coupling between modules.
* Prefer explicit data handoffs (dataframes, return values, or defined globals).
* Design modules so they can be **tested independently**.

---

### 2. Variable Management Rules

* **Consolidate all variables within the master pipeline block.**
* If variables are needed elsewhere, **import them as globals**.
* Do not scatter configuration values across cells.
* Avoid duplicate constants inside modules.
* Add new variables to the master pipeline first, then wire downstream.

---

### 3. Notes and Commentary

* **Preserve all notes exactly where they are placed.**
* Do not remove or rewrite notes unless explicitly instructed.
* Treat markdown and comments as long-term documentation.
* Flag outdated notes instead of deleting them.

---

### 4. Notebook Structure

Preferred order:

1. Preflight / setup
2. Authentication / mounts
3. Shared imports
4. Module sections
5. Validation / diagnostics
6. Export / outputs
7. Master pipeline
8. Utility / recovery helpers

Use clear section headers (e.g., `### Preflight Check`, `# Master Pipeline`).

---

### 5. Module Design Requirements

* Each module should have **one responsibility**.
* Wrap logic in clearly named functions.
* Define inputs and outputs explicitly.
* Avoid hidden dependencies.
* Include lightweight validation.
* Document side effects (exports, file moves, etc.).

---

## 6. Master Pipeline Responsibilities

The master pipeline must:

* Define all variables and configuration
* Control execution order
* Pass configuration to modules
* Handle global state intentionally
* Manage logging and status
* Coordinate exports and failure handling

---

### 7. Globals Usage Rules

* Use globals only for intentionally shared configuration.
* Assign globals in the master pipeline.
* Avoid implicit globals in modules.
* Make dependencies on globals explicit.

---

### 8. Imports and Dependencies

* Keep imports organized.
* Avoid unnecessary duplication.
* Include imports in modules only if needed for isolation.
* Do not introduce unnecessary libraries.

---

### 9. Validation and Debugging

* Add validation checkpoints after transformations.
* Include diagnostics (row counts, schema checks, etc.).
* Print clear status messages.
* Fail gracefully where possible.

---

### 10. Output and Export Standards

* Keep export logic in a dedicated section.
* Use clear, traceable naming conventions.
* Avoid hidden output paths.
* Document outputs clearly.

---

### 11. Recovery and Utility Cells

* Keep utilities separate from core workflow.
* Clearly label recovery logic.
* Preserve existing utility cells.

---

### 12. Change Management

* Preserve structure unless improvement is necessary.
* Do not collapse modular design.
* Prefer targeted edits.
* Explain structural changes when needed.

---

### 13. Coding Style

* Write readable, maintainable code.
* Use clear function names.
* Prefer explicit logic over shortcuts.
* Preserve dataframe clarity.

---

### 14. Interaction Rules

When building workflows:

* Assume modular architecture is required.
* Place variables in the master pipeline.
* Preserve notes and structure.
* Return code ready for direct cell insertion.
* Highlight impacted sections when making changes.

---

### Short Instruction Block (Reusable)

```text
Build this Colab workflow using a modular notebook architecture.

Rules:
1. Consolidate all variables within the master pipeline block.
2. Import shared variables as globals when needed.
3. Preserve all notes and markdown exactly as placed.
4. Keep each module self-contained and reusable.
5. Use a master pipeline for orchestration.
6. Do not scatter configuration values.
7. Maintain clear section headers.
8. Separate validation, export, and recovery logic.
9. Prefer targeted updates over rewrites.
10. Write maintainable, debuggable code.
```


### DATA EXTRACTION LOGIC

Do not use hard-coded row numbers or fixed positional logic when parsing these reports. Instead, use anchor-based detection by defining constants for known header or label text, such as const headers = ['Occ (%)', 'Index (MPI)', ...], and locate rows dynamically based on those anchors. This ensures the parser remains stable even if rows shift between properties, report versions, or months.

The parsing logic should always identify the relevant section by searching for the expected text labels in the sheet, rather than assuming a metric will always appear on the same row. Build the mapping from those discovered anchor points, then derive the related values relative to the matched labels. This makes the pipeline more resilient and reduces breakage when report formatting changes.

Use anchor-based parsing only. Never rely on fixed row indexes for report extraction. Define reusable constants for expected labels and headers, scan the sheet to find those anchors, and build mappings from the discovered positions. This ensures the parser continues to work even when report layouts shift.

# Success Critera Test

To deterime if the merge was successful without affecting the underlaying data integrity this test made and must pass before considering success.

Display a Dataframe with the following totals, filtered by Date + Reservations Status for the exported file.

---

## Filters
Filters
stay_date = `01-01-2026` - `01-31-2026`
status = `CHECKEDOUT`

---

## Sum Values
Sum Col `rms_otb`
Sum Col `rev_otb`

---

## Criteria
Total for `rms_otb` must equal **2235**
Total for `rev_otb` Revenue must equal **175872**

# Setup & Auth

In [ ]:
# @title Connect to Google Drive {"vertical-output":true,"single-column":true,"display-mode":"code"}

from google.colab import drive
import os

def setup_environment(source_path, next_path):
    """
    Module: Setup Environment
    Mounts Google Drive, defines global directory variables, and ensures all
    required subfolders exist.
    """
    print("--- Initializing Environment ---")

# --- SETUP & AUTH ---
# Use force_remount=True to attempt a fresh connection if it previously failed
try:
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully.")
except Exception as e:
    print(f"Manual action required: Please click the Drive icon in the left file pane to mount your drive. Error: {e}")

# --- CONFIGURATION (Master Variables) ---
global SOURCE_DIR, NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR

SOURCE_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step04" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/process_step04"}
NEXT_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step05" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/process_step05"}

NEW_DIR = os.path.join(SOURCE_DIR, "data_upload")
PROCESSED_DIR = os.path.join(SOURCE_DIR, "data_processed")
EXPORT_DIR = os.path.join(SOURCE_DIR, "data_export")
FAILED_DIR = os.path.join(SOURCE_DIR, "data_failed")
NEXT_DIR = os.path.join(NEXT_DIR, "data_upload")

# Create the necessary folders if they don't exist
if not os.path.exists(NEW_DIR):
    os.makedirs(NEW_DIR)
    print(f"Created directory: {NEW_DIR}")
if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)
    print(f"Created directory: {PROCESSED_DIR}")
if not os.path.exists(EXPORT_DIR):
    os.makedirs(EXPORT_DIR)
    print(f"Created directory: {EXPORT_DIR}")
if not os.path.exists(FAILED_DIR):
    os.makedirs(FAILED_DIR)
    print(f"Created directory: {FAILED_DIR}")

    # 4. Create directories dynamically if they don't exist
    directories = [NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR]

    for directory in directories:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"  -> Created directory: {directory}")

    print(f"v Setup complete. Checking for new files in: {NEW_DIR}\n")

Mounted at /content/drive
Drive mounted successfully.


# Start Code Replacement

In [ ]:
import os
import pandas as pd
import glob

# --- MODULE: DATA LOADING ---
def load_standardized_data(source_dir, pattern="*_standardized*"):
    """
    Searches for a standardized CSV in the source directory and loads it.
    Returns the DataFrame and the path of the loaded file, or (None, None) if not found.
    """
    search_glob = os.path.join(source_dir, "**", pattern)
    found_files = glob.glob(search_glob, recursive=True)
    found_files = [f for f in found_files if os.path.isfile(f)]

    if not found_files:
        print(f"x No files matching '{pattern}' found in {source_dir}")
        return None, None

    target_path = found_files[0]
    try:
        df = pd.read_csv(target_path, on_bad_lines='skip', low_memory=False)
        print(f"v Successfully loaded: {os.path.basename(target_path)}")
        return df, target_path
    except Exception as e:
        print(f"x Error loading {target_path}: {e}")
        return None, None

def find_header_row(file_path):
    """Reads the beginning of a file to find the row index where the header likely starts."""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if any(keyword in line.lower() for keyword in ['stay_date', 'confirmation', 'rate_code', 'room']):
                return i
    return 0

def drop_unnecessary_columns(df):
    """Removes specific internal and metadata columns. Targeted at snake_case library output."""
    if df is None:
        return None

    # List of columns to drop
    cols_to_drop = ['source_file_metadata', 'Unnamed: 0']

    # Identify columns that actually exist in the dataframe
    existing_cols = [c for c in cols_to_drop if c in df.columns]

    if existing_cols:
        df = df.drop(columns=existing_cols)
        print(f"  -> Successfully removed columns: {existing_cols}")

    return df

In [ ]:
import os, re
import glob # Added glob import
import pandas as pd # Added pandas import for pd.to_datetime and re.search


# --- DIAGNOSTIC MODULE ---
def check_drive_paths():

    global NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR

    print("--- Drive Diagnostic ---")
    if not os.path.exists('/content/drive'):
        print("x Google Drive is NOT mounted.")
        return

    print("v Google Drive is mounted.")
    sd_path = '/content/drive/Shareddrives'
    if os.path.exists(sd_path):
        print(f"\nShared Drives found:")
        for sd in os.listdir(sd_path):
            print(f" - '{sd}'")
    else:
        print("x Shared Drives folder not accessible.")

    # Test the specific target
    target = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step04"
    if os.path.exists(target):
        print(f"\nv Target path exists: {target}")
    else:
        print(f"\nx Target path NOT found: {target}")

check_drive_paths()

--- Drive Diagnostic ---
v Google Drive is mounted.

Shared Drives found:
 - 'Aparium'
 - 'Backup'
 - 'BCT: Creative Hub'
 - 'ClientHubs'
 - 'Confidential'
 - 'Creative'
 - 'Creative Hub'
 - 'Data'
 - 'External Files'
 - 'Finance'
 - 'Helpfiles'
 - 'Hosted'
 - 'Partners '
 - 'REVREBEL'
 - 'REVREBEL Wiki'
 - 'Stringham Family'
 - 'Templates'
 - 'The Library'
 - 'Toolkits'
 - 'Vault'

v Target path exists: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step04


# Display Mapping Tables


In [ ]:
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd

# Authenticate and create the client
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
spreadsheet_id = '1aqez7bFrAhyRWxwVGu5-06ZKwY9eXWbnwOnHyFkauBg'
tabs_to_load = [
    'map_rate',
    'map_crs_channel',
    'map_crs_subsource',
    'map_subsource',
    'map_pms_source',
    'map_segment',
    'map_channel',
    'map_pms_segment',
    'map_overides'
]

# Dictionary to store the DataFrames
dataframes = {}

spreadsheet = gc.open_by_key(spreadsheet_id)

for tab in tabs_to_load:
    try:
        worksheet = spreadsheet.worksheet(tab)
        data = worksheet.get_all_values()
        # Assume first row is header
        dataframes[tab] = pd.DataFrame(data[1:], columns=data[0])
        print(f"Successfully loaded tab: {tab}")
    except Exception as e:
        print(f"Error loading tab {tab}: {e}")

# Example: Access a specific dataframe
# display(dataframes['map_rate'].head())

Successfully loaded tab: map_rate
Successfully loaded tab: map_crs_channel
Successfully loaded tab: map_crs_subsource
Successfully loaded tab: map_subsource
Successfully loaded tab: map_pms_source
Successfully loaded tab: map_segment
Successfully loaded tab: map_channel
Successfully loaded tab: map_pms_segment
Successfully loaded tab: map_overides


In [ ]:
print(f"Total tables loaded: {len(dataframes)}")

for name in tabs_to_load:
    if name in dataframes:
        print(f'\n--- Table: {name} ---')
        print(f"Rows: {len(dataframes[name])}")
        display(dataframes[name].head())
    else:
        print(f"\n x Table '{name}' was not found in the loaded dictionary.")

Total tables loaded: 9

--- Table: map_rate ---
Rows: 123


,pms_ratecode,ratecode_name,segment_code,channel_code,channel,source_code,source,subsource_code,subsource
0,ADV,Plan Ahead & Save,UQ,,,,,,
1,BAR,Best Flexible,RE,,,,,,
2,BFCM,Black Friday / Cyber Monday,PR,,,,,,
3,BUDDY,Buddy Buddy Opening Promo,PR,,,,,,
4,CCRP1,CCRP1,UQ,,,,,,



--- Table: map_crs_channel ---
Rows: 20


,crs_channel,source_code,source,subsource_code,subsource
0,Mobile,BE,Booking Engine,MB,Mobile
1,Booking Engine,BE,Booking Engine,WB,Desktop
2,Dnata,DN,Dnata,DN,Dnata
3,Expedia,EG,Expedia Group,,
4,Sabre,GD,GDS,,



--- Table: map_crs_subsource ---
Rows: 39


...

,source_code,source,channel_code,channel,source_sort,channel_sort
0,HD,Hotel Direct,OP,On-Property,1,1
1,BE,Booking Engine,BE,Booking Engine,2,2
2,CR,Central Reservations,VO,Voice,4,3
3,VO,Voice,DC,Direct Connect,4,5
4,GD,GDS,GD,GDS,9,4



--- Table: map_pms_segment ---
Rows: 1


,pms_segment,segment_code,segment,segment_group_code,segment_group,segment_sort
0,Group,GR,Group,GGG,Group,30



--- Table: map_overides ---
Rows: 2


,crs_channel,source_code,source,subsource_code,subsource
0,goog,MS,Metasearch,GG,Google
1,goog_organic,MS,Metasearch,GG,Google


# Modules

In [ ]:
def apply_rate_mapping(df, mapping_dict):
    """
    Joins the input dataframe with the map_rate table.
    Updates: ratecode_name, segment_code, channel_code, channel, source_code, source, subsource_code, subsource
    """
    df = df.copy()
    if 'map_rate' not in mapping_dict: return df

    map_rate_df = mapping_dict['map_rate'].copy()
    map_rate_df['pms_ratecode'] = map_rate_df['pms_ratecode'].astype(str).str.strip()
    df['pms_ratecode'] = df['pms_ratecode'].astype(str).str.strip()
    map_rate_df = map_rate_df.drop_duplicates(subset=['pms_ratecode'], keep='first')

    cols_to_use = ['ratecode_name', 'segment_code', 'channel_code', 'channel', 'source_code', 'source', 'subsource_code', 'subsource']
    available_cols = [c for c in cols_to_use if c in map_rate_df.columns]

    # Use prefix to avoid collision
    rename_map = {c: f"map_rate_{c}" for c in available_cols}
    map_subset = map_rate_df[['pms_ratecode'] + available_cols].rename(columns=rename_map)

    merged = pd.merge(df, map_subset, on='pms_ratecode', how='left')

    for col in available_cols:
        map_col = f"map_rate_{col}"
        if col in merged.columns:
            merged[col] = merged[map_col].fillna(merged[col])
        else:
            merged[col] = merged[map_col]
        merged[col] = merged[col].fillna("")
        merged = merged.drop(columns=[map_col])

    return merged

In [ ]:
def apply_crs_mapping(df, mapping_dict):
    """
    Joins with map_crs_channel on 'crs_channel'.
    Updates 'source_code' and 'subsource_code' ONLY if they are currently blank/NaN.
    """
    df = df.copy()

    if 'map_crs_channel' not in mapping_dict:
        return df

    map_crs_df = mapping_dict['map_crs_channel'].copy()
    if 'crs_channel' not in df.columns:
        return df

    df['crs_channel'] = df['crs_channel'].astype(str).str.strip()
    map_crs_df['crs_channel'] = map_crs_df['crs_channel'].astype(str).str.strip()

    # Prepare mapping table
    cols_to_extract = ['crs_channel', 'source_code', 'source', 'subsource_code', 'subsource']
    available_map_cols = [c for c in cols_to_extract if c in map_crs_df.columns]

    # Use clear prefixes for the map data to avoid _x/_y suffixes during merge
    rename_map = {c: f"map_crs_{c}" for c in available_map_cols if c != 'crs_channel'}
    map_subset = map_crs_df[available_map_cols].rename(columns=rename_map)

    merged = pd.merge(df, map_subset, on='crs_channel', how='left')

    # Logic: Fill only if original is blank
    for col in ['source_code', 'source', 'subsource_code', 'subsource']:
        map_col = f"map_crs_{col}"
        if map_col in merged.columns:
            if col not in merged.columns:
                merged[col] = ""

            # Fill empty/NaN values with mapped values
            mask = (merged[col].astype(str).fillna("").str.strip() == "")
            merged.loc[mask, col] = merged.loc[mask, map_col]
            merged = merged.drop(columns=[map_col])

    return merged

In [ ]:
def apply_crs_subsource(df, mapping_dict):
    """
    Map subsource_code and subsource using crs_subsource_code.

    Existing values are preserved unless a valid mapping match exists.
    Blank mapping values do not overwrite existing values.

    Final output keeps blank cells as "" instead of pandas <NA>.
    """

    df = df.copy()

    mapping_name = "map_crs_subsource"
    join_key = "crs_subsource_code"
    target_cols = ["subsource_code", "subsource"]
    blank_values = ["", "nan", "none", "<na>", "null"]

    if mapping_name not in mapping_dict:
        print(f"Note: '{mapping_name}' not found. Skipping CRS subsource mapping.")
        return df

    if join_key not in df.columns:
        print(f"Warning: '{join_key}' not found in source data. Skipping.")
        return df

    map_df = mapping_dict[mapping_name].copy()

    if join_key not in map_df.columns:
        print(
            f"Warning: '{join_key}' not found in mapping "
            f"'{mapping_name}'. Skipping."
        )
        return df

    available_cols = [
        col for col in target_cols
        if col in map_df.columns
    ]

    if not available_cols:
        print(
            f"Note: No usable mapping columns found in "
            f"'{mapping_name}'. Skipping."
        )
        return df

    # Clean join keys for matching.
    # Use fillna("") so pandas does not create visible <NA> values.
    df[join_key] = (
        df[join_key]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    map_df[join_key] = (
        map_df[join_key]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Remove blank mapping keys so they cannot match blank source rows.
    map_df = map_df[
        map_df[join_key].str.lower().isin(blank_values) == False
    ].copy()

    # Prevent duplicate mapping keys from duplicating source rows.
    duplicate_mask = map_df.duplicated(
        subset=[join_key],
        keep=False
    )

    if duplicate_mask.any():
        duplicate_count = duplicate_mask.sum()

        print(
            f"Warning: {duplicate_count} duplicate mapping rows found "
            f"for '{join_key}'. Using the last mapping for each key."
        )

        map_df = map_df.drop_duplicates(
            subset=[join_key],
            keep="last"
        )

    # Make sure original destination columns exist and are normal text columns.
    for col in available_cols:
        if col not in df.columns:
            df[col] = ""
        else:
            df[col] = (
                df[col]
                .fillna("")
                .astype(str)
                .str.strip()
            )

            # Clean accidental text versions of missing values.
            df.loc[
                df[col].str.lower().isin(blank_values),
                col
            ] = ""

    # Clean mapping output columns.
    for col in available_cols:
        map_df[col] = (
            map_df[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        map_df.loc[
            map_df[col].str.lower().isin(blank_values),
            col
        ] = ""

    # Rename mapping columns so originals remain available.
    rename_map = {
        col: f"mapped_{col}"
        for col in available_cols
    }

    map_subset = (
        map_df[[join_key] + available_cols]
        .rename(columns=rename_map)
    )

    merged = df.merge(
        map_subset,
        on=join_key,
        how="left",
        validate="many_to_one"
    )

    matched_rows = pd.Series(
        False,
        index=merged.index
    )

    # Apply mapped values only when the mapping contains a real value.
    for col in available_cols:
        mapped_col = f"mapped_{col}"

        mapped_value = (
            merged[mapped_col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        valid_mapping = (
            mapped_value.ne("")
            & ~mapped_value.str.lower().isin(blank_values)
        )

        matched_rows = matched_rows | valid_mapping

        merged.loc[valid_mapping, col] = mapped_value.loc[valid_mapping]

    # Remove temporary mapping columns.
    merged = merged.drop(
        columns=[
            f"mapped_{col}"
            for col in available_cols
        ]
    )

    # Final cleanup: keep true blank cells as empty strings, not <NA>.
    for col in available_cols:
        merged[col] = (
            merged[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        merged.loc[
            merged[col].str.lower().isin(blank_values),
            col
        ] = ""

    print(
        f"CRS Subsource Mapping: "
        f"{matched_rows.sum()} / {len(merged)} rows "
        f"received mapped values."
    )

    return merged

In [ ]:
def apply_pms_source_mapping(df, mapping_dict):
    """
    Joins with map_pms_source on 'pms_source'.
    Updates 'source_code', 'source', 'subsource_code', and 'subsource' ONLY if they are currently blank/NaN.
    """
    df = df.copy()

    if 'map_pms_source' not in mapping_dict:
        print("x Warning: 'map_pms_source' not found in mapping dictionary. Skipping.")
        return df

    map_pms_df = mapping_dict['map_pms_source'].copy()

    # Standardize join key
    if 'pms_source' not in df.columns:
        print(f"i Note: 'pms_source' column not found in source data. Skipping PMS Source mapping.")
        return df

    df['pms_source'] = df['pms_source'].astype(str).str.strip()
    map_pms_df['pms_source'] = map_pms_df['pms_source'].astype(str).str.strip()

    # Deduplicate
    duplicate_count = map_pms_df.duplicated(subset=['pms_source'], keep=False).sum()
    if duplicate_count > 0:
        print(f"  -> Warning: {duplicate_count} duplicate keys found in map_pms_source. Keeping the first occurrence.")
        map_pms_df = map_pms_df.drop_duplicates(subset=['pms_source'], keep='first')

    potential_cols = ['source_code', 'source', 'subsource_code', 'subsource']
    cols_to_map = [c for c in potential_cols if c in map_pms_df.columns]

    if not cols_to_map:
        print("i Note: No matching target columns found in map_pms_source. Skipping.")
        return df

    # Prefix mapping columns to prevent overlap
    map_subset = map_pms_df[['pms_source'] + cols_to_map].rename(columns={c: f'map_{c}' for c in cols_to_map})

    # Left join
    merged = pd.merge(df, map_subset, on='pms_source', how='left')

    # Logic: Fill only if blank (Original wins)
    for col in cols_to_map:
        mapped_col = f'map_{col}'
        if col not in merged.columns:
            merged[col] = merged[mapped_col]
        else:
            merged[col] = merged[col].fillna(merged[mapped_col])
            mask = (merged[col].astype(str).str.strip() == "")
            merged.loc[mask, col] = merged.loc[mask, mapped_col]

        # Prevent "NaN" floats by filling with empty strings
        merged[col] = merged[col].fillna("")

    # Cleanup temporary columns
    merged = merged.drop(columns=[f'map_{c}' for c in cols_to_map])

    return merged

In [ ]:
def apply_subsource_mapping(df, mapping_dict):
    """
    Joins with map_subsource on 'subsource'.
    Updates 'subsource_code', 'subsource'
    """
    df = df.copy()

    if 'map_subsource' not in mapping_dict:
        print("x Warning: 'map_subsource' not found in mapping dictionary. Skipping.")
        return df

    map_pms_df = mapping_dict['map_subsource'].copy()

    # Standardize join key
    if 'subsource_code' not in df.columns:
        print(f"i Note: 'subsource_code' column not found in source data. Skipping PMS Source mapping.")
        return df

    df['subsource_code'] = df['subsource_code'].astype(str).str.strip()
    map_pms_df['subsource_code'] = map_pms_df['subsource_code'].astype(str).str.strip()

    # Deduplicate
    duplicate_count = map_pms_df.duplicated(subset=['subsource_code'], keep=False).sum()
    if duplicate_count > 0:
        print(f"  -> Warning: {duplicate_count} duplicate keys found in map_subsource. Keeping the first occurrence.")
        map_pms_df = map_pms_df.drop_duplicates(subset=['subsource_code'], keep='first')

    potential_cols = ['subsource']
    cols_to_map = [c for c in potential_cols if c in map_pms_df.columns]

    if not cols_to_map:
        print("i Note: No matching target columns found in map_subsource. Skipping.")
        return df

    # Prefix mapping columns to prevent overlap
    map_subset = map_pms_df[['subsource_code'] + cols_to_map].rename(columns={c: f'map_{c}' for c in cols_to_map})

    # Left join
    merged = pd.merge(df, map_subset, on='subsource_code', how='left')

    # Logic: Fill only if blank (Original wins)
    for col in cols_to_map:
        mapped_col = f'map_{col}'
        if col not in merged.columns:
            merged[col] = merged[mapped_col]
        else:
            merged[col] = merged[col].fillna(merged[mapped_col])
            mask = (merged[col].astype(str).str.strip() == "")
            merged.loc[mask, col] = merged.loc[mask, mapped_col]

        # Prevent "NaN" floats by filling with empty strings
        merged[col] = merged[col].fillna("")

    # Cleanup temporary columns
    merged = merged.drop(columns=[f'map_{c}' for c in cols_to_map])

    return merged

In [ ]:
def apply_channel_mapping(df, mapping_dict):
    """
    Joins with map_channel on 'source_code'.
    Updates: channel_code, channel, channel_sort, source, source_sort
    """
    df = df.copy()

    if 'map_channel' not in mapping_dict:
        print("i Note: 'map_channel' not found in mapping dictionary. Skipping.")
        return df

    map_chan_df = mapping_dict['map_channel'].copy()

    # Standardize join key
    if 'source_code' not in df.columns:
        print(f"i Note: 'source_code' column not found. Skipping Channel mapping.")
        return df

    df['source_code'] = df['source_code'].astype(str).str.strip()
    map_chan_df['source_code'] = map_chan_df['source_code'].astype(str).str.strip()

    # Deduplicate
    duplicate_count = map_chan_df.duplicated(subset=['source_code'], keep=False).sum()
    if duplicate_count > 0:
        print(f"  -> Warning: {duplicate_count} duplicate keys found in map_channel. Keeping the first occurrence.")
        map_chan_df = map_chan_df.drop_duplicates(subset=['source_code'], keep='first')

    cols_to_map = ['channel_code', 'channel', 'channel_sort', 'source', 'source_sort']
    available_cols = [c for c in cols_to_map if c in map_chan_df.columns]

    # Prefix mapping columns
    rename_map = {c: f"map_{c}" for c in available_cols}
    map_subset = map_chan_df[['source_code'] + available_cols].rename(columns=rename_map)

    # Left join
    merged = pd.merge(df, map_subset, on='source_code', how='left')

    # Update logic: Map wins, fallback to original, scrub NaNs
    for col in available_cols:
        map_col = f"map_{col}"
        if col in merged.columns:
            merged[col] = merged[map_col].fillna(merged[col])
        else:
            merged[col] = merged[map_col]

        merged[col] = merged[col].fillna("")

    # Cleanup temporary columns
    merged = merged.drop(columns=[f"map_{c}" for c in available_cols])

    return merged

In [ ]:
def apply_segment_mapping(df, mapping_dict):
    """
    Joins with map_segment on 'segment_code'.
    Updates: segment, segment_group_code, segment_group, segment_sort
    """
    df = df.copy()

    if 'map_segment' not in mapping_dict:
        print("i Note: 'map_segment' not found in mapping dictionary. Skipping.")
        return df

    map_seg_df = mapping_dict['map_segment'].copy()

    # Standardize join key
    if 'segment_code' not in df.columns:
        print(f"i Note: 'segment_code' column not found in source data. Skipping Segment mapping.")
        return df

    df['segment_code'] = df['segment_code'].astype(str).str.strip()
    map_seg_df['segment_code'] = map_seg_df['segment_code'].astype(str).str.strip()

    # Deduplicate
    duplicate_count = map_seg_df.duplicated(subset=['segment_code'], keep=False).sum()
    if duplicate_count > 0:
        print(f"  -> Warning: {duplicate_count} duplicate keys found in map_segment. Keeping the first occurrence.")
        map_seg_df = map_seg_df.drop_duplicates(subset=['segment_code'], keep='first')

    column_map = ['segment_group_code', 'segment_group', 'segment', 'segment_sort']
    available_cols = [c for c in column_map if c in map_seg_df.columns]

    # Prefix mapping columns
    rename_map = {c: f"map_{c}" for c in available_cols}
    map_subset = map_seg_df[['segment_code'] + available_cols].rename(columns=rename_map)

    # Left join
    merged = pd.merge(df, map_subset, on='segment_code', how='left')

    # Update logic: Map wins, fallback to original, scrub NaNs
    for col in available_cols:
        map_col = f"map_{col}"
        if col in merged.columns:
            merged[col] = merged[map_col].fillna(merged[col])
        else:
            merged[col] = merged[map_col]

        merged[col] = merged[col].fillna("")

    # Cleanup temporary columns
    merged = merged.drop(columns=[f"map_{c}" for c in available_cols])

    return merged

In [ ]:
def apply_pms_segment_mapping(df, mapping_dict):
    """
    Joins with map_pms_segment on 'pms_segment'.
    Updates: segment, segment_group_code, segment_group, segment_sort
    """
    df = df.copy()

    if 'map_pms_segment' not in mapping_dict:
        print("i Note: 'map_pms_segment' not found in mapping dictionary. Skipping.")
        return df

    map_seg_df = mapping_dict['map_pms_segment'].copy()

    # Standardize join key
    if 'pms_segment' not in df.columns:
        print(f"i Note: 'pms_segment' column not found in source data. Skipping Segment mapping.")
        return df

    df['pms_segment'] = df['pms_segment'].astype(str).str.strip()
    map_seg_df['pms_segment'] = map_seg_df['pms_segment'].astype(str).str.strip()

    # Deduplicate
    duplicate_count = map_seg_df.duplicated(subset=['pms_segment'], keep=False).sum()
    if duplicate_count > 0:
        print(f"  -> Warning: {duplicate_count} duplicate keys found in map_pms_segment. Keeping the first occurrence.")
        map_seg_df = map_seg_df.drop_duplicates(subset=['pms_segment'], keep='first')

    column_map = ['segment_code', 'segment', 'segment_group_code', 'segment_group', 'segment_sort']
    available_cols = [c for c in column_map if c in map_seg_df.columns]

    # Prefix mapping columns
    rename_map = {c: f"map_{c}" for c in available_cols}
    map_subset = map_seg_df[['pms_segment'] + available_cols].rename(columns=rename_map)

    # Left join
    merged = pd.merge(df, map_subset, on='pms_segment', how='left')

    # Update logic: Map wins, fallback to original, scrub NaNs
    for col in available_cols:
        map_col = f"map_{col}"
        if col in merged.columns:
            merged[col] = merged[map_col].fillna(merged[col])
        else:
            merged[col] = merged[map_col]

        merged[col] = merged[col].fillna("")

    # Cleanup temporary columns
    merged = merged.drop(columns=[f"map_{c}" for c in available_cols])

    return merged

In [ ]:
def apply_manual_overrides(df, mapping_dict):
    """
    Applies manual overrides using crs_channel.

    Only changes a field when:
    1. crs_channel is present in both datasets,
    2. crs_channel contains a real, non-blank value, and
    3. the override field contains a valid, non-blank value.

    Unmatched rows retain their existing values.
    """
    df = df.copy()

    mapping_name = "map_overides"
    join_key = "crs_channel"
    target_cols = [
        "source_code",
        "source",
        "subsource_code",
        "subsource"
    ]

    if mapping_name not in mapping_dict:
        print(f"Note: '{mapping_name}' not found. Skipping.")
        return df

    if join_key not in df.columns:
        print(f"Warning: '{join_key}' not found in source data. Skipping.")
        return df

    map_df = mapping_dict[mapping_name].copy()

    if join_key not in map_df.columns:
        print(f"Warning: '{join_key}' not found in '{mapping_name}'. Skipping.")
        return df

    available_cols = [
        col for col in target_cols
        if col in map_df.columns
    ]

    if not available_cols:
        print(f"Note: No override columns found in '{mapping_name}'. Skipping.")
        return df

    # Preserve true missing values rather than converting them to "nan".
    df[join_key] = df[join_key].astype("string").str.strip()
    map_df[join_key] = map_df[join_key].astype("string").str.strip()

    # Blank override keys must never be allowed to match.
    map_df = map_df[
        map_df[join_key].notna()
        & map_df[join_key].ne("")
    ].copy()

    # Use one override record per channel.
    duplicate_count = map_df.duplicated(join_key, keep=False).sum()

    if duplicate_count:
        print(
            f"Warning: {duplicate_count} duplicate manual override rows found "
            f"for '{join_key}'. Using the last row for each key."
        )
        map_df = map_df.drop_duplicates(join_key, keep="last")

    rename_map = {
        col: f"override_{col}"
        for col in available_cols
    }

    override_subset = map_df[
        [join_key] + available_cols
    ].rename(columns=rename_map)

    merged = df.merge(
        override_subset,
        on=join_key,
        how="left",
        validate="many_to_one"
    )

    row_match = merged[
        [f"override_{col}" for col in available_cols]
    ].notna().any(axis=1)

    changed_cells = 0

    for col in available_cols:
        override_col = f"override_{col}"
        override_value = merged[override_col]

        valid_override = (
            override_value.notna()
            & override_value.astype("string").str.strip().ne("")
        )

        if col not in merged.columns:
            merged[col] = pd.NA

        changed_cells += valid_override.sum()

        merged.loc[valid_override, col] = merged.loc[
            valid_override,
            override_col
        ]

    merged = merged.drop(
        columns=[f"override_{col}" for col in available_cols]
    )

    print(
        f"Manual Overrides: {row_match.sum()} / {len(merged)} rows matched; "
        f"{changed_cells} individual values replaced."
    )

    return merged

In [ ]:
# --- MODULE: VALIDATION & DIAGNOSTICS ---
def run_data_audit(df):
    """
    Calculates completion statistics for key categorization columns.
    """
    target_headers = [
        'channel_code', 'channel', 'channel_sort',
        'segment_code', 'segment', 'segment_sort',
        'source_code', 'source', 'source_sort',
        'subsource_code', 'subsource'
    ]

    # Filter headers that actually exist in the dataframe
    existing_headers = [h for h in target_headers if h in df.columns]

    stats = []
    for col in existing_headers:
        total_rows = len(df)
        # Count non-null and non-empty strings
        populated_count = df[col].apply(lambda x: str(x).strip() != "" and pd.notnull(x) and str(x).lower() != 'nan').sum()
        empty_count = total_rows - populated_count

        stats.append({
            'Column': col,
            'Total Rows': total_rows,
            'Populated': populated_count,
            'Empty / NaN': empty_count,
            '% Populated': f"{(populated_count / total_rows) * 100:.2f}%"
        })

    audit_df = pd.DataFrame(stats)
    return audit_df

In [ ]:
import datetime

# --- MODULE: EXPORT ---
def export_to_csv(df, export_dir, filename_prefix="processed_reservations"):
    """
    Exports the dataframe to a CSV file in the specified directory.
    """
    if not os.path.exists(export_dir):
        os.makedirs(export_dir)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{filename_prefix}_{timestamp}.csv"
    export_path = os.path.join(export_dir, filename)

    try:
        df.to_csv(export_path, index=False)
        print(f"v Successfully exported: {filename}")
        print(f"i Location: {export_path}")
        return export_path
    except Exception as e:
        print(f"x Export failed: {e}")
        return None

In [ ]:
def apply_expedia_overrides(df):
    """
    Module: Expedia Fallback
    Applies 'EX' / 'Expedia' labels when source_code is 'EG' and subsource is empty.
    """
    df = df.copy()

    # Identify rows where source_code is EG and subsource_code is visually empty
    source_is_eg = (
        df["source_code"]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
        .eq("EG")
    )

    subsource_code_is_empty = (
        df["subsource_code"]
        .astype("string")
        .fillna("")
        .str.strip()
        .eq("")
    )

    mask = source_is_eg & subsource_code_is_empty

    # Ensure columns are string type before assignment
    df["subsource_code"] = df["subsource_code"].astype("string")
    df["subsource"] = df["subsource"].astype("string")

    # Apply fallback values
    df.loc[mask, ["subsource_code", "subsource"]] = ["EX", "Expedia"]

    print(f"i Expedia Fallback: Applied to {mask.sum()} rows out of {source_is_eg.sum()} total 'EG' rows.")

    return df




def apply_group_name_override(df):
    """
    Module: Group Name Override
    If pms_segment is 'Group', copies the value from group_name into ratecode_name.
    """
    df = df.copy()

    # 1. Check if the required columns exist in the dataframe
    required_cols = ['pms_segment', 'group_name', 'ratecode_name']
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        print(f"i Note: Missing columns {missing_cols}. Skipping Group override.")
        return df

    # 2. Find rows where pms_segment is 'Group' (case-insensitive, ignoring extra spaces)
    is_group = df['pms_segment'].astype("string").str.strip().str.lower().eq("group")

    # 3. Make sure group_name actually has a valid value to copy
    has_group_name = (
        df['group_name'].notna()
        & df['group_name'].astype("string").str.strip().ne("")
        & ~df['group_name'].astype("string").str.lower().isin(["nan", "none", "<na>"])
    )

    # 4. Combine conditions: It must be a Group AND have a valid group_name
    update_mask = is_group & has_group_name

    # 5. Apply the values
    df.loc[update_mask, 'ratecode_name'] = df.loc[update_mask, 'group_name']

    # Diagnostic print
    print(f"i Group Override: Copied group_name to ratecode_name for {update_mask.sum()} rows.")

    return df



def apply_subsource_name_overrides(df):
    """
    Module: Subsource Name Override
    Populates the 'subsource' column with specific names based on the 'subsource_code'.
    """
    df = df.copy()

    # 1. Check if the required columns exist
    if 'subsource_code' not in df.columns:
        print("i Note: 'subsource_code' column missing. Skipping Subsource Name override.")
        return df

    if 'subsource' not in df.columns:
        # Create it if it somehow doesn't exist yet
        df['subsource'] = pd.NA

    # 2. Define your specific overrides here
    override_map = {
        'BK': 'Booking',
        'AG': 'Agoda',
        'CT': 'Trip'
    }

    # 3. Clean the subsource_code column to ensure accurate matching (uppercase and strip spaces)
    clean_codes = df['subsource_code'].astype("string").str.strip().str.upper()

    # 4. Find the rows where the code exists in our dictionary
    update_mask = clean_codes.isin(override_map.keys())

    # 5. Apply the mapped values to the 'subsource' column
    df.loc[update_mask, 'subsource'] = clean_codes.map(override_map)

    # Diagnostic print
    print(f"i Subsource Name Override: Updated 'subsource' for {update_mask.sum()} rows.")

    return df

# Main Pipeline

In [ ]:
import pandas as pd
import os
import glob
from google.colab import drive
import pandas_gbq
import shutil # Import shutil for file operations

# --- EXECUTE SETUP ---
# setup_environment definition is located in the Preflight section (Y66b4AMqXFwD)
setup_environment(
    source_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step04",
    next_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step05"
)
def run_pipeline():

    global SOURCE_DIR, NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR

    """
    Master Pipeline Orchestrator.
    Executes all mapping, auditing, and exporting steps.
    """
    global reservations_df

    if 'SOURCE_DIR' not in globals():
        print("x Error: Run setup cell first.")
        return

    source_data, loaded_file_path = load_standardized_data(NEW_DIR)

    if source_data is None:
        print(f"x Pipeline Halted: Source file not found.")
        return

    # Extract metadata from filename
    file_name = os.path.basename(loaded_file_path)
    date_match = re.search(r"(\d{8})", file_name)
    last_processed_date = date_match.group(1) if date_match else "UNKNOWN"
    prop_match = re.search(r"([A-Z]{4,6})", file_name)
    last_processed_prop = prop_match.group(1).upper() if prop_match else "UNKNOWN"

    # 0. Column Removal
    current_count = len(source_data)
    print(f"\nStep 0: Pre-processing (Drop Columns) - Start Rows: {current_count}")
    if 'drop_unnecessary_columns' in globals():
        source_data = drop_unnecessary_columns(source_data)
    processed_df = source_data
    print(f"Step 0: Complete - End Rows: {len(processed_df)}")

    # 1. Rate Mapping
    current_count = len(processed_df)
    print(f"\nStep 1: Applying Rate Mapping - Start Rows: {current_count}")
    processed_df = apply_rate_mapping(processed_df, dataframes)
    print(f"Step 1: Complete - End Rows: {len(processed_df)}")

    # 2. CRS Channel Mapping
    current_count = len(processed_df)
    print(f"\nStep 2: Applying CRS Mapping (Channel) - Start Rows: {current_count}")
    if 'map_crs_channel' in dataframes:
        processed_df = apply_crs_mapping(processed_df, dataframes)
    print(f"Step 2: Complete - End Rows: {len(processed_df)}")

    # 3. CRS Subsource Mapping
    current_count = len(processed_df)
    print(f"\nStep 3: Applying CRS Mapping (Subsource) - Start Rows: {current_count}")
    if 'map_crs_subsource' in dataframes:
        processed_df = apply_crs_subsource(processed_df, dataframes)
    print(f"Step 3: Complete - End Rows: {len(processed_df)}")

    # 3a. CRS Subsource Mapping
    current_count = len(processed_df)
    print(f"\nStep 3a: Applying CRS Mapping (Subsource) - Start Rows: {current_count}")
    if 'map_subsource' in dataframes:
        processed_df = apply_subsource_mapping(processed_df, dataframes)
    print(f"Step 3: Complete - End Rows: {len(processed_df)}")


    # 4. PMS Source Mapping
    current_count = len(processed_df)
    print(f"\nStep 4: Applying PMS Source Mapping - Start Rows: {current_count}")
    if 'map_pms_source' in dataframes:
        processed_df = apply_pms_source_mapping(processed_df, dataframes)
    print(f"Step 4: Complete - End Rows: {len(processed_df)}")

    # 5. Segment Mapping
    current_count = len(processed_df)
    print(f"\nStep 5: Applying Segment Mapping - Start Rows: {current_count}")
    if 'map_segment' in dataframes:
        processed_df = apply_segment_mapping(processed_df, dataframes)
    print(f"Step 5: Complete - End Rows: {len(processed_df)}")

    # 6. Channel Mapping
    current_count = len(processed_df)
    print(f"\nStep 6: Applying Channel Mapping - Start Rows: {current_count}")
    if 'map_channel' in dataframes:
        processed_df = apply_channel_mapping(processed_df, dataframes)
    print(f"Step 6: Complete - End Rows: {len(processed_df)}")

    # 7. Manual Overrides
    current_count = len(processed_df)
    print(f"\nStep 7: Applying Manual Overrides - Start Rows: {current_count}")
    if 'map_overides' in dataframes:
        processed_df = apply_manual_overrides(processed_df, dataframes)
    print(f"Step 7: Complete - End Rows: {len(processed_df)}")

    # 8. PMS Segment Mapping
    current_count = len(processed_df)
    print(f"\nStep 8: PMS Segment Mapping - Start Rows: {current_count}")
    if 'map_pms_segment' in dataframes:
        processed_df = apply_pms_segment_mapping(processed_df, dataframes)
    print(f"Step 8: Complete - End Rows: {len(processed_df)}")

    # 9a. Expedia Fallback
    current_count = len(processed_df)
    print(f"\nStep 9a: Applying Expedia Fallback - Start Rows: {current_count}")
    processed_df = apply_expedia_overrides(processed_df)
    print(f"Step 9a: Complete - End Rows: {len(processed_df)}")

    # 9b Group Name Override
    current_count = len(processed_df)
    print(f"\nStep 9b: Applying Group Name Override - Start Rows: {current_count}")
    if 'apply_group_name_override' in globals():
        processed_df = apply_group_name_override(processed_df)
    print(f"Step 9b: Complete - End Rows: {len(processed_df)}")

    # 9c Subsource Name Override
    current_count = len(processed_df)
    print(f"\nStep 9c: Applying Subsource Name Override - Start Rows: {current_count}")
    if 'apply_subsource_name_overrides' in globals():
        processed_df = apply_subsource_name_overrides(processed_df)
    print(f"Step 9c: Complete - End Rows: {len(processed_df)}")

    reservations_df = processed_df

    # 10. Audit & Export
    print("\n--- FINAL DATA QUALITY AUDIT ---")
    audit_results = run_data_audit(reservations_df)
    display(audit_results)

    print("\nStep 10: Exporting Final Results...")

    output_filename = f"{last_processed_date}_{last_processed_prop}_standardized_data.csv"
    export_path = os.path.join(EXPORT_DIR, output_filename)
    next_step_path = os.path.join(NEXT_DIR, output_filename)

    reservations_df.to_csv(export_path, index=False)
    reservations_df.to_csv(next_step_path, index=False)

    # Move source file to PROCESSED_DIR
    shutil.move(loaded_file_path, os.path.join(PROCESSED_DIR, os.path.basename(loaded_file_path)))
    print(f"✅ Source file moved to processed: {os.path.basename(loaded_file_path)}")


    # Triggers the next notebook in the pipeline
    %run '/content/drive/MyDrive/Colab Notebooks/StayInTouch/Step05_StayInTouch_CreateUniqueColMetrics.ipynb'

    print(f"\nSUCCESS! Modular export complete: {output_filename}")
    print(f"Final Dataset row count: {len(reservations_df)}")
    display(reservations_df.head())

--- Initializing Environment ---


In [ ]:
# Execute the Master Pipeline defined in cell 0b1c2a9d
run_pipeline()

v Successfully loaded: 20260707_JFKNOW_headers_standardized_data.csv

Step 0: Pre-processing (Drop Columns) - Start Rows: 173292
Step 0: Complete - End Rows: 173292

Step 1: Applying Rate Mapping - Start Rows: 173292
Step 1: Complete - End Rows: 173292

Step 2: Applying CRS Mapping (Channel) - Start Rows: 173292
Step 2: Complete - End Rows: 173292

Step 3: Applying CRS Mapping (Subsource) - Start Rows: 173292
CRS Subsource Mapping: 32192 / 173292 rows received mapped values.
Step 3: Complete - End Rows: 173292

Step 3a: Applying CRS Mapping (Subsource) - Start Rows: 173292
Step 3: Complete - End Rows: 173292

Step 4: Applying PMS Source Mapping - Start Rows: 173292
Step 4: Complete - End Rows: 173292

Step 5: Applying Segment Mapping - Start Rows: 173292
Step 5: Complete - End Rows: 173292

Step 6: Applying Channel Mapping - Start Rows: 173292
Step 6: Complete - End Rows: 173292

Step 7: Applying Manual Overrides - Start Rows: 173292
Manual Overrides: 0 / 173292 rows matched; 0 individ

,Column,Total Rows,Populated,Empty / NaN,% Populated
0,channel_code,173292,173067,225,99.87%
1,channel,173292,173067,225,99.87%
2,channel_sort,173292,173037,255,99.85%
3,segment_code,173292,173282,10,99.99%
4,segment,173292,173282,10,99.99%
5,segment_sort,173292,173282,10,99.99%
6,source_code,173292,173067,225,99.87%
7,source,173292,173067,225,99.87%
8,source_sort,173292,173037,255,99.85%
9,subsource_code,173292,173062,230,99.87%



Step 10: Exporting Final Results...
Listing contents of: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step04/data_upload
- 20260707_JFKNOW_headers_standardized_data.csv
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step05/data_upload
--- Initializing Environment ---
Mounted at /content/drive
v Drive mounted successfully.
v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step05/data_upload

Found 1 files in /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step05/data_upload.

Processing: 20260707_JFKNOW_standardized_data.csv
i Unique Metrics: Created 'unique_lead_days' (cleared 133384 duplicate rows).
i Unique Metrics: Created 'unique_nights' (cleared 133384 duplicate rows).
✅ Schema Enfo

2it [00:25, 12.90s/it]


✅ Success: Data uploaded to BigQuery table: dovetailco.stg.pms_reservations
Moved original file to PROCESSED_DIR.

Master pipeline finished.
Loading latest export for validation: 20260707_JFKNOW_standardized_data.csv
--- Running Success Criteria Test ---
Filtering based on column: stay_date


,Metric,Actual,Target,Status
0,Sum Col 'rms_otb',2235.00,2235,✅ PASS
1,Sum Col 'rev_otb',175872.66,175872,✅ PASS



✨ SUCCESS: All criteria met. Data integrity confirmed.

SUCCESS! Modular export complete: 20260707_JFKNOW_standardized_data.csv
Final Dataset row count: 173292


,stay_date,confirmation_number,rate,original_rate,status,roomtype,room_no,block_count,travel_agent,company,...,segment,segment_sort,source_code,source,source_sort,subsource_code,subsource,ratecode_name,segment_group_code,segment_group
0,2024-12-09,100069.0,0.0,0.0,CHECKEDOUT,WSC,353.0,NaN,Direct Booking,NaN,...,Transient Unqualified,16,HD,Hotel Direct,1,HD,Hotel Direct,Plan Ahead & Save,TUQ,Transient Discount
1,2025-01-01,100000.0,100.0,199.0,CANCELED,ADA,NaN,NaN,NaN,NaN,...,Transient Retail,11,HD,Hotel Direct,1,HD,Hotel Direct,Best Flexible,TRE,Transient Retail
2,2025-01-02,100000.0,100.0,139.0,CANCELED,ADA,NaN,NaN,NaN,NaN,...,Transient Retail,11,HD,Hotel Direct,1,HD,Hotel Direct,Best Flexible,TRE,Transient Retail
3,2025-01-07,100197.0,0.0,0.0,CHECKEDOUT,WSC,348.0,NaN,Direct Booking,NaN,...,Transient Unqualified,16,HD,Hotel Direct,1,HD,Hotel Direct,Plan Ahead & Save,TUQ,Transient Discount
4,2025-01-08,100198.0,0.0,0.0,CHECKEDOUT,WSC,318.0,NaN,NaN,NaN,...,Transient Unqualified,16,HD,Hotel Direct,1,HD,Hotel Direct,Plan Ahead & Save,TUQ,Transient Discount


In [ ]:
# --- VERIFICATION: PREVIEW STEP 9a BEHAVIOR ---
# This manually runs the Expedia Fallback on the Step 5 results to confirm it fixes the <NA> issue.

if 'reservations_df' in locals():
    print("Previewing result of applying Step 9a (Expedia Fallback) to current debug data:")
    test_df = apply_expedia_overrides(reservations_df)

    # Filter to show only the Expedia rows that were just updated
    expedia_check = test_df[test_df['source_code'].astype(str).str.upper() == 'EG']
    display(expedia_check[['confirmation_number', 'source_code', 'subsource_code', 'subsource']].head(10))
else:
    print("reservations_df not found. Please run the pipeline cell first.")

# Validation / Diagnostics
This section verifies the integrity of the merged data against the success criteria defined for January 2026.

In [ ]:
import pandas as pd
import os
import glob



# --- Validation Module ---
def run_success_criteria_test(df):
    print("--- Running Success Criteria Test ---")

    # Filtering strictly on the 'Date' column as requested
    date_col = 'stay_date'

    if date_col not in df.columns:
         print(f"❌ Error: Required column '{date_col}' not found in the dataset.")
         return

    print(f"Filtering based on column: {date_col}")
    df[date_col] = pd.to_datetime(df[date_col])

    # 1. Define Filters (Criteria uhfOwqak033W)
    start_date = '2026-01-01'
    end_date = '2026-01-31'
    status_filter = 'CHECKEDOUT'

    # 2. Apply Filters
    mask = (
        (df[date_col] >= start_date) &
        (df[date_col] <= end_date) &
        (df['status'] == status_filter)
    )
    test_df = df.loc[mask].copy()

    # 3. Calculate Sums
    total_sold = pd.to_numeric(test_df['rms_otb'], errors='coerce').sum()
    total_revenue = pd.to_numeric(test_df['rev_otb'], errors='coerce').sum()

    # 4. Success Criteria Targets
    target_sold = 2235
    target_revenue = 175872

    # 5. Display Results with Tolerance for Decimals
    sold_pass = int(round(total_sold)) == target_sold
    rev_pass = abs(total_revenue - target_revenue) < 1.0

    results_data = {
        "Metric": ["Sum Col 'rms_otb'", "Sum Col 'rev_otb'"],
        "Actual": [round(total_sold, 2), round(total_revenue, 2)],
        "Target": [target_sold, target_revenue],
        "Status": [
            "✅ PASS" if sold_pass else "❌ FAIL",
            "✅ PASS" if rev_pass else "❌ FAIL"
        ]
    }

    results_df = pd.DataFrame(results_data)
    display(results_df)

    if sold_pass and rev_pass:
        print("\n✨ SUCCESS: All criteria met. Data integrity confirmed.")
    else:
        print("\n⚠️ WARNING: Criteria mismatch. Check if the dates or filters need adjustment.")

# Execution Logic
if 'combined_df' in locals():
    run_success_criteria_test(combined_df)
elif 'EXPORT_DIR' in globals():
    files = glob.glob(os.path.join(EXPORT_DIR, "*_standardized_data.csv"))
    if files:
        latest_export = max(files, key=os.path.getmtime)
        print(f"Loading latest export for validation: {os.path.basename(latest_export)}")
        loaded_df = pd.read_csv(latest_export, low_memory=False)
        run_success_criteria_test(loaded_df)

        # --- TRIGGER DOWNSTREAM PIPELINE ---
        print("\n🚀 Triggering Step 04...")
        get_ipython().run_line_magic('run', "'/content/drive/MyDrive/Colab Notebooks/StayInTouch/Step05_StayInTouch_CreateUniqueColMetrics.ipynb'")

    else:
        print("❌ Error: No exported files found in EXPORT_DIR.")
else:
    print("❌ Error: combined_df not found and EXPORT_DIR not defined.")